# LSTM Code Summarization

This notebook implements a sequence-to-sequence LSTM model for Java code summarization. The goal is to generate short natural-language summaries for Java methods. The workflow includes dataset preparation, embedding generation using CodeT5, LSTM training, validation analysis, and final evaluation on a held-out test set using BLEU, METEOR, BERTScore, and SIDE.

In [20]:
!pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement torch==2.0.1 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==2.0.1


In [21]:
!pip uninstall -y transformers tokenizers
!pip install transformers==4.38.2 tokenizers==0.15.2

Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: tokenizers 0.15.2
Uninstalling tokenizers-0.15.2:
  Successfully uninstalled tokenizers-0.15.2
  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.38.2-py3-none-any.whl (8.5 MB)
Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [22]:
!pip install -q javalang gitpython pandas requests bert-score nltk sacrebleu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.7 MB/s eta 0:00:00


In [23]:
import os
import re
import ast
import sys
import json
import time
import random
import shutil
import subprocess
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import javalang
import nltk
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sacrebleu.metrics import BLEU
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bertscore_score
from transformers import AutoTokenizer, AutoModel
sys.setrecursionlimit(20000)

In [24]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [26]:
folders = [
    "processed_data",
    "artifacts",
    "checkpoints",
    "outputs"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Directories ready")

Directories ready


In [27]:
required_files = [
    "get_codet5_embeddings.py",
    "requirements.txt",
    "sample_code.txt",
    "sample_summary.txt"
]

for f in required_files:
    print(f, "exists:", os.path.exists(f))

get_codet5_embeddings.py exists: True
requirements.txt exists: True
sample_code.txt exists: True
sample_summary.txt exists: True


In [28]:
!python get_codet5_embeddings.py \
    --input sample_code.txt \
    --output artifacts/sample_code.pt

Loading tokenizer and model: Salesforce/codet5p-220m
Traceback (most recent call last):
  File "/content/get_codet5_embeddings.py", line 71, in <module>
    main()
  File "/content/get_codet5_embeddings.py", line 27, in main
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 693, in from_pretrained
    return tokenizer_class_from_name(tokenizer_config_class).from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1721, in from_pretrained
    return cls._from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1910, in _from_pretrained
    tokenizer = cls(*init_inputs, **init_kwargs)
                ^^^^^^

In [29]:
!python get_codet5_embeddings.py \
    --input sample_summary.txt \
    --output artifacts/sample_summary.pt \
    --max_length 128

Loading tokenizer and model: Salesforce/codet5p-220m
Traceback (most recent call last):
  File "/content/get_codet5_embeddings.py", line 71, in <module>
    main()
  File "/content/get_codet5_embeddings.py", line 27, in main
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 693, in from_pretrained
    return tokenizer_class_from_name(tokenizer_config_class).from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1721, in from_pretrained
    return cls._from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1910, in _from_pretrained
    tokenizer = cls(*init_inputs, **init_kwargs)
                ^^^^^^

In [30]:
sample_code_data = torch.load("artifacts/sample_code.pt")
sample_summary_data = torch.load("artifacts/sample_summary.pt")

In [31]:
embedding_layer = nn.Embedding.from_pretrained(
    sample_code_data["embedding_matrix"],
    freeze=False
)

In [32]:
code_token_ids = sample_code_data["token_ids"]
summary_token_ids = sample_summary_data["token_ids"]

pad_id = sample_code_data["pad_token_id"]
eos_id = sample_code_data["eos_token_id"]
vocab_size = sample_code_data["vocab_size"]
embedding_dim = sample_code_data["embedding_dim"]

## Dataset Preparation

The training data was constructed by mining public GitHub Java repositories. Java source files were collected and parsed to extract method-level code snippets. When available, nearby documentation comments were used as natural-language summaries for each method.

The dataset pipeline performed several preprocessing steps:

- filtering non-Java files and irrelevant directories
- extracting method bodies from Java files
- pairing methods with nearby documentation comments
- flattening each code snippet into a single whitespace-normalized line
- lowercasing summaries
- removing duplicates

The resulting dataset was saved as parallel text files:

- `processed_data/train_code.txt`
- `processed_data/train_summary.txt`
- `processed_data/val_code.txt`
- `processed_data/val_summary.txt`

Each line in a code file corresponds to the summary on the same line in the paired summary file.

In [33]:
CLONE_DIR = "/content/java_repos"
OUTPUT_DIR = "/content/processed_data"

NUM_REPOS = 800
STARS_THRESHOLD = 50
FILES_PER_REPO = 60
TRAIN_TARGET = 50000
VAL_SIZE = 1000
MIN_METHOD_CHARS = 40
MAX_METHOD_CHARS = 4000
MIN_SUMMARY_WORDS = 3
MAX_SUMMARY_WORDS = 30
SEED = 42

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

EXCLUDE_PATTERNS = [
    "test", "tests", "example", "examples", "sample",
    "demo", "generated", "build", "target", "out"
]

random.seed(SEED)
os.makedirs(CLONE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete")
print("CLONE_DIR:", CLONE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

# gitHub repo fetching
def fetch_top_java_repos(
    num_repos: int = 300,
    per_page: int = 100,
    stars_threshold: int = 50,
    token: Optional[str] = None
) -> List[Dict]:
    repos = []
    page = 1
    url = "https://api.github.com/search/repositories"

    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    query = f"language:java stars:>{stars_threshold} fork:false archived:false"

    while len(repos) < num_repos:
        params = {
            "q": query,
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }

        r = requests.get(url, params=params, headers=headers, timeout=60)
        if r.status_code != 200:
            print("GitHub API error:", r.status_code, r.text[:300])
            break

        items = r.json().get("items", [])
        if not items:
            break

        for item in items:
            repos.append({
                "rank": len(repos) + 1,
                "full_name": item["full_name"],
                "clone_url": item["clone_url"],
                "stars": item["stargazers_count"]
            })
            if len(repos) >= num_repos:
                break

        page += 1
        time.sleep(0.5)

    return repos[:num_repos]

# cloning
def clone_repo(clone_url: str, dest_dir: str, timeout_s: int = 120) -> bool:
    if os.path.exists(dest_dir):
        return True
    try:
        cmd = ["git", "clone", "--depth", "1", "--quiet", clone_url, dest_dir]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        print(f"Timeout cloning {clone_url}")
        return False
    except Exception as e:
        print(f"Clone error for {clone_url}: {e}")
        return False

# file discovery / reading
def read_file_content(file_path: str) -> Optional[str]:
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            with open(file_path, "r", encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
        except Exception:
            return None
    return None

def find_java_files(repo_path: str) -> List[str]:
    java_files = []
    for root, dirs, files in os.walk(repo_path):
        root_lower = root.lower()
        if any(p in root_lower for p in EXCLUDE_PATTERNS):
            continue
        for f in files:
            if f.endswith(".java"):
                java_files.append(os.path.join(root, f))
    return java_files

def select_java_files(java_files: List[str], max_files: int) -> List[str]:
    if len(java_files) <= max_files:
        return java_files
    return random.sample(java_files, max_files)

# method source extraction
def extract_method_source(method_node, lines: List[str]) -> Optional[str]:
    try:
        if method_node.position is None:
            return None

        start_line = method_node.position.line - 1
        if start_line < 0 or start_line >= len(lines):
            return None

        brace_count = 0
        started = False
        end_line = start_line

        for i in range(start_line, len(lines)):
            line = lines[i]
            for ch in line:
                if ch == "{":
                    brace_count += 1
                    started = True
                elif ch == "}":
                    brace_count -= 1

            if started and brace_count == 0:
                end_line = i
                break

        method_text = "\n".join(lines[start_line:end_line + 1]).strip()
        if not method_text.endswith("}"):
            return None
        return method_text
    except Exception:
        return None

# javadoc/comment extraction
def clean_summary_text(text: str) -> str:
    text = re.sub(r"/\*\*?", " ", text)
    text = re.sub(r"\*/", " ", text)
    text = re.sub(r"^\s*\*\s?", " ", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*//\s?", " ", text, flags=re.MULTILINE)

    # remove common doc tags
    text = re.sub(r"@\w+\b.*", " ", text)
    text = re.sub(r"\{[@#].*?\}", " ", text)
    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    # keep first sentence when possible
    m = re.match(r"(.+?[.!?])(\s|$)", text)
    if m:
        text = m.group(1)

    text = text.strip(" .-").lower()
    return text


def extract_preceding_comment(lines: List[str], method_start_line: int) -> Optional[str]:
    """
    Accept:
      - Javadoc /** ... */
      - block comments /* ... */
      - consecutive // comment lines
    directly above a method, allowing blank lines / annotations in between.
    """
    i = method_start_line - 1

    # skip blank lines and annotations above method
    while i >= 0 and (lines[i].strip() == "" or lines[i].strip().startswith("@")):
        i -= 1

    if i < 0:
        return None

    line = lines[i].strip()

    # block or javadoc comment ending with */
    if "*/" in line:
        comment_lines = []
        found_start = False
        while i >= 0:
            comment_lines.append(lines[i])
            if "/*" in lines[i]:
                found_start = True
                break
            i -= 1

        if found_start:
            comment_lines.reverse()
            raw = "\n".join(comment_lines).strip()
            summary = clean_summary_text(raw)
            return summary if summary else None

    # one or more // lines
    if line.startswith("//"):
        comment_lines = []
        while i >= 0 and lines[i].strip().startswith("//"):
            comment_lines.append(lines[i])
            i -= 1
        comment_lines.reverse()
        raw = "\n".join(comment_lines).strip()
        summary = clean_summary_text(raw)
        return summary if summary else None

    return None

# pair filtering
def normalize_code(code: str) -> str:
    return re.sub(r"\s+", " ", code).strip()

def is_valid_summary(summary: str) -> bool:
    if not summary:
        return False

    words = summary.split()
    if len(words) < 2 or len(words) > 40:
        return False

    bad_phrases = {
        "todo", "fixme", "deprecated", "generated by",
        "auto generated", "do not modify"
    }
    if any(b in summary for b in bad_phrases):
        return False

    if re.fullmatch(r"[\W_]+", summary):
        return False

    return True

def is_valid_method(code: str) -> bool:
    if not code:
        return False
    if len(code) < MIN_METHOD_CHARS or len(code) > MAX_METHOD_CHARS:
        return False
    if " class " in code and code.count("{") <= 1:
        return False
    if code.strip().startswith("//"):
        return False
    return True

# main extraction per file
def extract_method_summary_pairs_from_file(file_path: str, repo_name: str) -> List[Dict]:
    pairs = []
    source_code = read_file_content(file_path)
    if source_code is None:
        return pairs

    lines = source_code.split("\n")

    try:
        tree = javalang.parse.parse(source_code)
    except RecursionError:
        return pairs
    except Exception:
        return pairs

    try:
        for _, node in tree.filter(javalang.tree.MethodDeclaration):
            try:
                method_source = extract_method_source(node, lines)
                if method_source is None:
                    continue

                if node.position is None:
                    continue

                summary = extract_preceding_comment(lines, node.position.line - 1)
                if summary is None:
                    continue

                code = normalize_code(method_source)
                summary = summary.lower().strip()

                if not is_valid_method(code):
                    continue
                if not is_valid_summary(summary):
                    continue

                pairs.append({
                    "repo": repo_name,
                    "file": file_path,
                    "method_name": node.name,
                    "code": code,
                    "summary": summary
                })
            except RecursionError:
                continue
            except Exception:
                continue
    except RecursionError:
        return pairs

    return pairs

def process_files_to_pairs(selected_files: List[Tuple[str, str]]) -> List[Dict]:
    all_pairs = []

    for idx, (repo_name, fp) in enumerate(selected_files):
        if idx % 1000 == 0 and idx > 0:
            print(f"Processed {idx}/{len(selected_files)} files... pairs so far: {len(all_pairs)}")

        try:
            file_pairs = extract_method_summary_pairs_from_file(fp, repo_name)
            all_pairs.extend(file_pairs)
        except RecursionError:
            continue
        except Exception:
            continue

    # dedup by exact code-summary pair
    seen_pair = set()
    dedup_pairs = []
    for x in all_pairs:
        key = (x["code"], x["summary"])
        if key not in seen_pair:
            seen_pair.add(key)
            dedup_pairs.append(x)

    # dedup by code only
    seen_code = set()
    final_pairs = []
    for x in dedup_pairs:
        if x["code"] not in seen_code:
            seen_code.add(x["code"])
            final_pairs.append(x)

    return final_pairs

def save_parallel_txt(df: pd.DataFrame, code_path: str, summary_path: str):
    with open(code_path, "w", encoding="utf-8") as fc, open(summary_path, "w", encoding="utf-8") as fs:
        for _, row in df.iterrows():
            fc.write(row["code"].strip() + "\n")
            fs.write(row["summary"].strip() + "\n")

# run pipeline
print("\n[1/6] Fetching repos from GitHub...")
repo_data = fetch_top_java_repos(
    num_repos=NUM_REPOS,
    per_page=100,
    stars_threshold=STARS_THRESHOLD,
    token=GITHUB_TOKEN
)
df_repos = pd.DataFrame(repo_data)
print("Fetched repos:", len(df_repos))

print("\n[2/6] Cloning repos...")
cloned_repos = []
failed_repos = []

for _, row in df_repos.iterrows():
    repo_name = row["full_name"]
    clone_url = row["clone_url"]
    safe_name = repo_name.replace("/", "_")
    dest_dir = os.path.join(CLONE_DIR, safe_name)

    ok = clone_repo(clone_url, dest_dir, timeout_s=120)
    if ok:
        cloned_repos.append({
            "repo_name": repo_name,
            "local_path": dest_dir,
            "stars": int(row["stars"])
        })
    else:
        failed_repos.append(repo_name)

print(f"Cloned: {len(cloned_repos)} | Failed: {len(failed_repos)}")

print("\n[3/6] Selecting Java files per repo...")
all_selected_files = []

for info in cloned_repos:
    repo_name = info["repo_name"]
    repo_path = info["local_path"]

    java_files = find_java_files(repo_path)
    if not java_files:
        continue

    selected = select_java_files(java_files, FILES_PER_REPO)
    all_selected_files.extend([(repo_name, f) for f in selected])

print("Total selected .java files:", len(all_selected_files))

print("\n[4/6] Extracting method-summary pairs...")
pairs = process_files_to_pairs(all_selected_files)
df_pairs = pd.DataFrame(pairs)

print("Pairs extracted:", len(df_pairs))
if len(df_pairs) == 0:
    raise ValueError("No method-summary pairs found. Increase NUM_REPOS / FILES_PER_REPO or relax filters.")

# shuffle
df_pairs = df_pairs.sample(frac=1, random_state=SEED).reset_index(drop=True)

# cap / split
needed_total = TRAIN_TARGET + VAL_SIZE
if len(df_pairs) < needed_total:
    print(f"Warning: only {len(df_pairs)} pairs found, less than target {needed_total}.")
    print("You can raise NUM_REPOS and/or FILES_PER_REPO and rerun.")
    train_df = df_pairs.iloc[VAL_SIZE:].copy()
    val_df = df_pairs.iloc[:min(VAL_SIZE, len(df_pairs))].copy()
else:
    df_pairs = df_pairs.iloc[:needed_total].copy()
    val_df = df_pairs.iloc[:VAL_SIZE].copy()
    train_df = df_pairs.iloc[VAL_SIZE:].copy()

print("\n[5/6] Saving train/validation text files...")
train_code_path = os.path.join(OUTPUT_DIR, "train_code.txt")
train_summary_path = os.path.join(OUTPUT_DIR, "train_summary.txt")
val_code_path = os.path.join(OUTPUT_DIR, "val_code.txt")
val_summary_path = os.path.join(OUTPUT_DIR, "val_summary.txt")

save_parallel_txt(train_df, train_code_path, train_summary_path)
save_parallel_txt(val_df, val_code_path, val_summary_path)

metadata = {
    "num_repos_requested": NUM_REPOS,
    "stars_threshold": STARS_THRESHOLD,
    "files_per_repo": FILES_PER_REPO,
    "train_target": TRAIN_TARGET,
    "val_size": VAL_SIZE,
    "pairs_extracted_total": int(len(df_pairs)),
    "train_pairs_saved": int(len(train_df)),
    "val_pairs_saved": int(len(val_df)),
    "failed_repos_count": len(failed_repos),
    "failed_repos_sample": failed_repos[:20],
    "sources_used": sorted(train_df["repo"].dropna().unique().tolist())[:200]
}
with open(os.path.join(OUTPUT_DIR, "dataset_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\n[6/6] Done")
print("Saved:")
print(" ", train_code_path)
print(" ", train_summary_path)
print(" ", val_code_path)
print(" ", val_summary_path)
print(" ", os.path.join(OUTPUT_DIR, "dataset_metadata.json"))

print("\nSample rows:")
display(train_df[["repo", "method_name", "summary"]].head(10))

Setup complete
CLONE_DIR: /content/java_repos
OUTPUT_DIR: /content/processed_data

[1/6] Fetching repos from GitHub...
GitHub API error: 403 {"message":"API rate limit exceeded for 34.125.233.103. (But here's the good news: Authenticated requests get a higher rate limit. Check out the documentation for more details.)","documentation_url":"https://docs.github.com/rest/overview/resources-in-the-rest-api#rate-limiting"}

Fetched repos: 500

[2/6] Cloning repos...
Timeout cloning https://github.com/Archmage83/tvapk.git
Cloned: 499 | Failed: 1

[3/6] Selecting Java files per repo...
Total selected .java files: 24439

[4/6] Extracting method-summary pairs...
Processed 1000/24439 files... pairs so far: 1183
Processed 2000/24439 files... pairs so far: 2669
Processed 3000/24439 files... pairs so far: 3841
Processed 4000/24439 files... pairs so far: 4787
Processed 5000/24439 files... pairs so far: 5664
Processed 6000/24439 files... pairs so far: 7047
Processed 7000/24439 files... pairs so far: 7

,repo,method_name,summary
1000,amitshekhariitbhu/Fast-Android-Networking,patch,method to make patch request
1001,Konloch/bytecode-viewer,resetWorkSpace,this will ask the user if they really want to ...
1002,vavr-io/vavr,getLeaf,"fetch the leaf, corresponding to the given index"
1003,clojure/clojure,readStream,reads the given input stream and returns its c...
1004,lightbend/config,systemEnvironment,gets a config containing the system's environm...
1005,apache/flink,createLeaderRetrievalDriverFactory,creates a {
1006,nayuki/QR-Code-generator,drawFinderPattern,draws a 9*9 finder pattern including the borde...
1007,getActivity/AndroidProject,removeViews,移除 textview 监听，避免内存泄露
1008,justauth/JustAuth,getRequestToken,obtaining a request token https://developer.tw...
1009,koush/ion,getAllMostSpecificFirst,parses the dn and returns all values for an at...


In [34]:
shutil.rmtree("/content/java_repos", ignore_errors=True)

print("Deleted cloned repositories")

Deleted cloned repositories


In [35]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   45G   69G  40% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
/dev/sda1       119G  111G  8.8G  93% /opt/bin/.nvidia
tmpfs           6.4G  4.5M  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [36]:
def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

train_code = read_lines("processed_data/train_code.txt")
train_summary = read_lines("processed_data/train_summary.txt")
val_code = read_lines("processed_data/val_code.txt")
val_summary = read_lines("processed_data/val_summary.txt")

print("Train code:", len(train_code))
print("Train summary:", len(train_summary))
print("Val code:", len(val_code))
print("Val summary:", len(val_summary))
print("Total pairs:", len(train_code) + len(val_code))

Train code: 30592
Train summary: 30592
Val code: 1000
Val summary: 1000
Total pairs: 31592


In [1]:
!python get_codet5_embeddings.py \
    --input processed_data/train_code.txt \
    --output artifacts/train_code.pt \
    --max_length 256

python3: can't open file '/content/get_codet5_embeddings.py': [Errno 2] No such file or directory


In [38]:
!python get_codet5_embeddings.py \
    --input processed_data/train_summary.txt \
    --output artifacts/train_summary.pt \
    --max_length 128

Loading tokenizer and model: Salesforce/codet5p-220m
Traceback (most recent call last):
  File "/content/get_codet5_embeddings.py", line 71, in <module>
    main()
  File "/content/get_codet5_embeddings.py", line 27, in main
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 693, in from_pretrained
    return tokenizer_class_from_name(tokenizer_config_class).from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1721, in from_pretrained
    return cls._from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1910, in _from_pretrained
    tokenizer = cls(*init_inputs, **init_kwargs)
                ^^^^^^

In [39]:
!python get_codet5_embeddings.py \
    --input processed_data/val_code.txt \
    --output artifacts/val_code.pt \
    --max_length 256

Loading tokenizer and model: Salesforce/codet5p-220m
Traceback (most recent call last):
  File "/content/get_codet5_embeddings.py", line 71, in <module>
    main()
  File "/content/get_codet5_embeddings.py", line 27, in main
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 693, in from_pretrained
    return tokenizer_class_from_name(tokenizer_config_class).from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1721, in from_pretrained
    return cls._from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1910, in _from_pretrained
    tokenizer = cls(*init_inputs, **init_kwargs)
                ^^^^^^

In [40]:
!python get_codet5_embeddings.py \
    --input processed_data/val_summary.txt \
    --output artifacts/val_summary.pt \
    --max_length 128

Loading tokenizer and model: Salesforce/codet5p-220m
Traceback (most recent call last):
  File "/content/get_codet5_embeddings.py", line 71, in <module>
    main()
  File "/content/get_codet5_embeddings.py", line 27, in main
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 693, in from_pretrained
    return tokenizer_class_from_name(tokenizer_config_class).from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1721, in from_pretrained
    return cls._from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1910, in _from_pretrained
    tokenizer = cls(*init_inputs, **init_kwargs)
                ^^^^^^

In [41]:
# load saved embedding artifacts
train_code_data = torch.load("artifacts/train_code.pt")
train_summary_data = torch.load("artifacts/train_summary.pt")
val_code_data = torch.load("artifacts/val_code.pt")
val_summary_data = torch.load("artifacts/val_summary.pt")

# shared embedding info
embedding_layer = nn.Embedding.from_pretrained(
    train_code_data["embedding_matrix"],
    freeze=False
)

pad_id = train_code_data["pad_token_id"]
eos_id = train_code_data["eos_token_id"]
vocab_size = train_code_data["vocab_size"]
embedding_dim = train_code_data["embedding_dim"]

print("Vocab size:", vocab_size)
print("Embedding dim:", embedding_dim)
print("Pad ID:", pad_id)
print("EOS ID:", eos_id)

# convert lists to tensors
from torch.nn.utils.rnn import pad_sequence

def pad_token_lists(token_lists, pad_id):
    tensors = [torch.tensor(x, dtype=torch.long) for x in token_lists]
    return pad_sequence(tensors, batch_first=True, padding_value=pad_id)

train_code_ids = pad_token_lists(train_code_data["token_ids"], pad_id)
train_summary_ids = pad_token_lists(train_summary_data["token_ids"], pad_id)

val_code_ids = pad_token_lists(val_code_data["token_ids"], pad_id)
val_summary_ids = pad_token_lists(val_summary_data["token_ids"], pad_id)

print("Train code shape:", train_code_ids.shape)
print("Train summary shape:", train_summary_ids.shape)
print("Val code shape:", val_code_ids.shape)
print("Val summary shape:", val_summary_ids.shape)


class CodeSummaryDataset(Dataset):
    def __init__(self, code_ids, summary_ids):
        self.code_ids = code_ids
        self.summary_ids = summary_ids

    def __len__(self):
        return len(self.code_ids)

    def __getitem__(self, idx):
        return {
            "src": self.code_ids[idx],
            "tgt": self.summary_ids[idx]
        }


train_dataset = CodeSummaryDataset(train_code_ids, train_summary_ids)
val_dataset = CodeSummaryDataset(val_code_ids, val_summary_ids)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

sample_batch = next(iter(train_loader))
print("Sample src batch shape:", sample_batch["src"].shape)
print("Sample tgt batch shape:", sample_batch["tgt"].shape)

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/train_code.pt'

## Model Architecture

The model is a sequence-to-sequence LSTM designed to generate natural-language summaries from Java code.

The architecture contains:

- an **encoder LSTM** that processes the tokenized Java method code
- a **decoder LSTM** that generates the summary tokens
- a **linear output layer** that projects decoder hidden states to the vocabulary space

The model uses embeddings initialized from CodeT5 (`Salesforce/codet5p-220m`). These embeddings were generated using the provided embedding script and loaded directly into the LSTM embedding layer.

In [ ]:
class LSTMCodeSummarizer(nn.Module):
    def __init__(self, embedding_layer, hidden_dim, vocab_size, pad_id, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = embedding_layer
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size
        self.pad_id = pad_id
        self.num_layers = num_layers

        embed_dim = embedding_layer.embedding_dim

        self.encoder = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.decoder = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.output_proj = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src_ids, tgt_ids):
        src_emb = self.embedding(src_ids)
        _, (h, c) = self.encoder(src_emb)

        dec_input = tgt_ids[:, :-1]
        dec_emb = self.embedding(dec_input)

        dec_out, _ = self.decoder(dec_emb, (h, c))
        logits = self.output_proj(dec_out)

        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMCodeSummarizer(
    embedding_layer=embedding_layer,
    hidden_dim=512,
    vocab_size=vocab_size,
    pad_id=pad_id,
    num_layers=1,
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Model ready.")

## Training Procedure

The model was trained using teacher forcing, where the decoder receives the previous ground-truth summary token as input during training. Cross-entropy loss was used as the training objective, with padding tokens ignored in the loss calculation.

The optimizer used was Adam. Training and validation losses were recorded each epoch to monitor learning progress and detect overfitting.

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in loader:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        optimizer.zero_grad()

        logits = model(src, tgt)

        tgt_out = tgt[:, 1:]

        loss = criterion(
            logits.reshape(-1, vocab_size),
            tgt_out.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            src = batch["src"].to(device)
            tgt = batch["tgt"].to(device)

            logits = model(src, tgt)

            tgt_out = tgt[:, 1:]

            loss = criterion(
                logits.reshape(-1, vocab_size),
                tgt_out.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):

    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = eval_epoch(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}")
    print("Train loss:", train_loss)
    print("Val loss:", val_loss)
    print()

## Early Stopping and Overfitting Analysis

Training loss steadily decreased across epochs, showing that the LSTM successfully learned patterns from the training data. Validation loss improved during the early epochs but eventually plateaued and slightly increased while training loss continued decreasing.

This divergence between training and validation loss suggests the model began to overfit the training data after the middle epochs. Based on the validation loss trend, early stopping would select a checkpoint from an earlier epoch as the best model for generalization.

####Epoch 1
Train loss: 4.812425705467559

Val loss: 4.117616459727287

####Epoch 2
Train loss: 3.742069862179959

Val loss: 3.657087594270706

####Epoch 3
Train loss: 3.1826902959899486

Val loss: 3.4103008285164833

####Epoch 4
Train loss: 2.763126751032386

Val loss: 3.2916988506913185

####Epoch 5
Train loss: 2.4173571908915483

Val loss: 3.213061183691025

####Epoch 6
Train loss: 2.1330311445542325

Val loss: 3.1910766661167145

####Epoch 7
Train loss: 1.889071451865433

Val loss: 3.1806343123316765

####Epoch 8
Train loss: 1.6832939132904974

Val loss: 3.210748068988323

####Epoch 9
Train loss: 1.5150325857562783

Val loss: 3.2347574681043625

####Epoch 10
Train loss: 1.3675939400663755

Val loss: 3.274309054017067


In [ ]:
torch.save(model.state_dict(), "checkpoints/lstm_epoch10.pt")
print("Saved current model.")

In [ ]:
def generate_summary(model, src_ids, max_len=128):
    model.eval()

    src_ids = src_ids.unsqueeze(0).to(device)  # [1, S]

    with torch.no_grad():
        src_emb = model.embedding(src_ids)
        _, (h, c) = model.encoder(src_emb)

        # start decoder with EOS token as a simple start token
        dec_input = torch.tensor([[eos_id]], device=device)

        generated = []

        for _ in range(max_len):
            dec_emb = model.embedding(dec_input)
            dec_out, (h, c) = model.decoder(dec_emb, (h, c))
            logits = model.output_proj(dec_out[:, -1, :])   # [1, vocab_size]

            next_token = torch.argmax(logits, dim=-1)       # [1]
            token_id = next_token.item()

            if token_id == eos_id:
                break

            generated.append(token_id)
            dec_input = next_token.unsqueeze(0)

    return generated

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")

def decode_tokens(token_ids):
    return tokenizer.decode(token_ids, skip_special_tokens=True).strip()

In [ ]:
predictions = []
references = []

for i in range(len(val_dataset)):

    src = val_dataset[i]["src"]
    tgt = val_dataset[i]["tgt"]

    pred_ids = generate_summary(model, src, max_len=40)

    pred_text = decode_tokens(pred_ids)
    ref_text = decode_tokens(tgt.tolist())

    predictions.append(pred_text)
    references.append(ref_text)

print("Generated", len(predictions), "predictions")

In [ ]:
bleu = BLEU(effective_order=True)

score = bleu.corpus_score(predictions, [references])

print("BLEU:", score.score)

In [ ]:
os.makedirs("outputs", exist_ok=True)

pred_df = pd.DataFrame({
    "prediction": predictions,
    "reference": references
})

pred_df.to_csv("outputs/val_predictions.csv", index=False)
print("Saved to outputs/val_predictions.csv")

In [ ]:
def compute_bleu_scores(predictions, references):
    scores = {}
    for n in [1, 2, 3, 4]:
        metric = BLEU(effective_order=True, max_ngram_order=n)
        scores[f"BLEU-{n}"] = metric.corpus_score(predictions, [references]).score
    return scores

bleu_scores = compute_bleu_scores(predictions, references)

for name, score in bleu_scores.items():
    print(f"{name}: {score:.4f}")

In [ ]:
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")

In [ ]:
fmeteor_scores = []
for pred, ref in zip(predictions, references):
    meteor_scores.append(meteor_score([ref.split()], pred.split()))

avg_meteor = sum(meteor_scores) / len(meteor_scores)
print("METEOR:", avg_meteor)

In [ ]:
P, R, F1 = bertscore_score(predictions, references, lang="en", verbose=True)
print("BERTScore F1:", F1.mean().item())

In [ ]:
!unzip -o 141205.zip

In [ ]:
os.listdir("141205")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_folder = "141205"

side_tokenizer = AutoTokenizer.from_pretrained(checkpoint_folder)
side_model = AutoModel.from_pretrained(checkpoint_folder).to(DEVICE)

side_model.eval()

print("SIDE model loaded.")

In [ ]:
val_code_texts = read_lines("processed_data/val_code.txt")

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

In [ ]:
def compute_side(code_text, summary_text):

    encoded = side_tokenizer(
        [code_text, summary_text],
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        output = side_model(**encoded)

    embeddings = mean_pooling(output, encoded["attention_mask"])
    embeddings = F.normalize(embeddings, p=2, dim=1)

    score = F.cosine_similarity(
        embeddings[0].unsqueeze(0),
        embeddings[1].unsqueeze(0)
    ).item()

    return score

In [ ]:
print(compute_side(val_code_texts[0], predictions[0]))

In [ ]:
side_scores = []

for code, pred in zip(val_code_texts, predictions):
    side_scores.append(compute_side(code, pred))

SIDE = sum(side_scores) / len(side_scores)
print("SIDE:", SIDE)

In [ ]:
final_metrics = {
    "BLEU-1": 15.1808,
    "BLEU-2": 9.3340,
    "BLEU-3": 6.5390,
    "BLEU-4": 5.2061,
    "METEOR": 0.1392134861899552,
    "BERTScore_F1": 0.8363414406776428,
    "SIDE": 0.3894703766554594
}

for k, v in final_metrics.items():
    print(f"{k}: {v}")

In [ ]:
test_df = pd.read_csv("test_dataset_tokenized.csv")

print("Test size:", len(test_df))
test_df.head()

In [ ]:
test_df["code_ids"] = test_df["code_ids"].apply(ast.literal_eval)

## Final Test Set Evaluation

The final trained model was evaluated on the provided held-out test set. This dataset was not used during training, validation, hyperparameter tuning, or early stopping. It was used only once for final reporting of evaluation metrics.

For each test example, the model generated a summary from the Java method code. The generated summaries were evaluated using the following metrics:

- BLEU-1, BLEU-2, BLEU-3, BLEU-4
- METEOR
- BERTScore
- SIDE

## Note on Test Set Size

The assignment description refers to a test set of 1,000 examples. However, the provided test dataset used in this notebook contains 99 code–summary pairs. All final evaluation metrics reported below were therefore computed on these 99 examples.

In [ ]:
test_predictions = []

for ids in test_df["code_ids"]:

    src_ids = torch.tensor(ids[:256], dtype=torch.long)

    pred_ids = generate_summary(model, src_ids)

    pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True).strip()

    test_predictions.append(pred_text)

print("Generated:", len(test_predictions))

In [ ]:
test_references = test_df["summary"].tolist()

In [ ]:
bleu_scores = compute_bleu_scores(test_predictions, test_references)

for name, score in bleu_scores.items():
    print(f"{name}: {score:.4f}")

In [ ]:
meteor_scores = []

for pred, ref in zip(test_predictions, test_references):
    meteor_scores.append(meteor_score([ref.split()], pred.split()))

print("METEOR:", sum(meteor_scores) / len(meteor_scores))

In [ ]:
P, R, F1 = bertscore_score(test_predictions, test_references, lang="en")
print("BERTScore F1:", F1.mean().item())

In [ ]:
test_codes = test_df["code"].tolist()

side_scores = []

for code, pred in zip(test_codes, test_predictions):
    side_scores.append(compute_side(code, pred))

TEST_SIDE = sum(side_scores) / len(side_scores)

print("SIDE:", TEST_SIDE)

In [ ]:
test_df["prediction"] = test_predictions

test_df[["code", "summary", "prediction"]].to_csv(
    "outputs/test_predictions.csv",
    index=False
)

print("Saved outputs/test_predictions.csv")

In [ ]:
print("Final Test Results")
print("------------------")

print("BLEU-1:", bleu_scores["BLEU-1"])
print("BLEU-2:", bleu_scores["BLEU-2"])
print("BLEU-3:", bleu_scores["BLEU-3"])
print("BLEU-4:", bleu_scores["BLEU-4"])
print("METEOR:", sum(meteor_scores) / len(meteor_scores))
print("BERTScore:", F1.mean().item())
print("SIDE:", TEST_SIDE)

## Results

The LSTM baseline produces modest performance on the code summarization task. Exact-match metrics such as BLEU and METEOR are relatively low, which is expected because small wording differences between generated and reference summaries significantly affect these metrics.

BERTScore is higher because it measures semantic similarity rather than exact token overlap. The SIDE score is lower, suggesting that the generated summaries do not always align strongly with the semantic meaning of the code. Overall, the model functions as a baseline system but shows the limitations of a simple LSTM architecture for this task.